# NewsBot 2.0 Final Project

**ITAI 2373 - Final Project**  
**Student:** Viktoriya Kurmisheva  
**Foundation:** Midterm NewsBot Intelligence System  
**Peer Review Format:** Team Testers with Abraham Barreto

This notebook transforms the midterm NewsBot into a more advanced news analysis platform. It keeps the working midterm foundation and adds final-project modules for topic modeling, summarization, semantic search, multilingual processing, conversational queries, and an advanced research extension.

The notebook uses free/open-source tools only. No paid API keys are required.

## 1. Environment Setup and Imports

This section installs or loads the libraries needed for the final system. The code uses free tools that work in Google Colab: pandas, scikit-learn, NLTK, spaCy, NetworkX, and optional Gradio.

In [ ]:
import sys
import subprocess
import importlib.util
from pathlib import Path


def package_available(import_name):
    """Return True when a Python package can be imported."""
    return importlib.util.find_spec(import_name) is not None


def install_if_missing(import_name, pip_name=None):
    """Install a package only when it is missing."""
    if not package_available(import_name):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


# Most Colab runtimes already include many of these packages, so this usually runs quickly.
for import_name, pip_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("nltk", "nltk"),
    ("spacy", "spacy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("networkx", "networkx"),
    ("langdetect", "langdetect"),
    ("wordcloud", "wordcloud"),
    ("gradio", "gradio"),
    ("IPython", "ipython"),
]:
    install_if_missing(import_name, pip_name)


import os
import re
import math
import json
import zipfile
import random
import warnings
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

import spacy
from spacy.language import Language

from langdetect import detect, DetectorFactory

from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity


warnings.filterwarnings("ignore")
DetectorFactory.seed = 42
random.seed(42)
np.random.seed(42)

for resource in [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "vader_lexicon",
]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("Environment ready.")

## 2. Configuration and Data Loading

The final project builds on the same BBC News Classification data used in the midterm. The dataset has substantial English articles, clear category labels, and more than 500 rows across five categories.

If this notebook runs in Colab, upload `learn-ai-bbc.zip` when prompted. If the file already exists in the notebook folder, the loader uses it automatically.

In [ ]:
class NewsBot2Config:
    """Central configuration for NewsBot 2.0."""

    def __init__(self):
        self.project_name = "NewsBot 2.0"
        self.random_state = 42
        self.max_articles = 2000
        self.min_article_chars = 200
        self.tfidf_features = 5000
        self.topic_features = 4000
        self.topic_count = 8
        self.entity_article_limit = 700
        self.default_summary_sentences = 3


config = NewsBot2Config()


def running_in_colab():
    """Detect whether the notebook is running in Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def locate_or_upload_bbc_zip():
    """Find the BBC dataset ZIP or ask the user to upload it in Colab."""
    candidates = [
        Path("learn-ai-bbc.zip"),
        Path("/content/learn-ai-bbc.zip"),
        Path.home() / "Downloads" / "learn-ai-bbc.zip",
        Path(r"C:\Users\Admin\Downloads\learn-ai-bbc.zip"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    if running_in_colab():
        from google.colab import files
        print("Upload learn-ai-bbc.zip from your computer.")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                return Path(name)

    raise FileNotFoundError("Could not find learn-ai-bbc.zip. Upload it or place it in the notebook folder.")


def load_bbc_dataset():
    """Load, standardize, and validate the BBC News Classification dataset."""
    data_dir = Path("bbc_final_data")
    data_dir.mkdir(exist_ok=True)

    zip_path = locate_or_upload_bbc_zip()
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(data_dir)

    csv_candidates = list(data_dir.rglob("BBC News Train.csv")) + list(data_dir.rglob("*Train*.csv"))
    if not csv_candidates:
        raise FileNotFoundError("Could not find the BBC training CSV after extracting the ZIP.")

    raw_df = pd.read_csv(csv_candidates[0])
    df = raw_df.rename(
        columns={"ArticleId": "article_id", "Text": "content", "Category": "category"}
    )[["article_id", "content", "category"]].copy()

    df["content"] = df["content"].fillna("").astype(str).str.strip()
    df["category"] = df["category"].fillna("").astype(str).str.strip()
    df["title"] = (
        df["content"]
        .str.split(r"\s{2,}|\n", n=1)
        .str[0]
        .str.split()
        .str[:16]
        .str.join(" ")
        .str.title()
    )
    df["source"] = "BBC News Classification (Kaggle)"

    # The BBC file does not include publication dates. This generated date supports trend demos only.
    df["analysis_date"] = pd.date_range("2024-01-01", periods=len(df), freq="D")
    df["analysis_month"] = df["analysis_date"].dt.to_period("M").astype(str)

    df = df[(df["content"].str.len() >= config.min_article_chars) & (df["category"].str.len() > 0)].copy()
    if len(df) > config.max_articles:
        df = df.sample(n=config.max_articles, random_state=config.random_state).copy()

    requirements = {
        "At least 500 articles": len(df) >= 500,
        "At least 4 categories": df["category"].nunique() >= 4,
        "At most 2,000 articles": len(df) <= 2000,
        "No missing content": df["content"].notna().all() and df["content"].str.len().gt(0).all(),
        "No missing category": df["category"].notna().all() and df["category"].str.len().gt(0).all(),
        "Substantial article text": df["content"].str.len().ge(config.min_article_chars).all(),
    }

    display(pd.DataFrame({"Requirement": requirements.keys(), "Passed": requirements.values()}))
    assert all(requirements.values()), "Dataset validation failed."
    return df.reset_index(drop=True)


df = load_bbc_dataset()
print(f"Loaded {len(df)} articles across {df['category'].nunique()} categories.")
display(df.head())

## 3. Enhanced Preprocessing Pipeline

This section keeps the midterm cleaning logic but wraps it into a reusable class. The final system uses the same preprocessing across classification, topic modeling, semantic search, and summarization.

In [ ]:
class TextPreprocessor:
    """Clean and normalize raw news text for NLP modeling."""

    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))

    def clean_text(self, text):
        """Remove URLs, HTML, email addresses, symbols, digits, and extra spacing."""
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
        text = re.sub(r"\S+@\S+", " ", text)
        text = re.sub(r"[^a-zA-Z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def tokenize(self, text):
        """Tokenize with a lightweight regex so the pipeline remains stable in Colab."""
        return re.findall(r"[a-z]+", text.lower())

    def preprocess(self, text, remove_stopwords=True, lemmatize=True):
        """Run the full preprocessing pipeline and return processed text."""
        cleaned = self.clean_text(text)
        tokens = self.tokenize(cleaned)
        if remove_stopwords:
            tokens = [token for token in tokens if token not in self.stop_words]
        if lemmatize:
            tokens = [self.lemmatizer.lemmatize(token) for token in tokens]
        tokens = [token for token in tokens if len(token) > 2]
        return " ".join(tokens)


preprocessor = TextPreprocessor()

df["full_text"] = df["title"] + " " + df["content"]
df["content_clean"] = df["content"].apply(preprocessor.clean_text)
df["content_processed"] = df["content"].apply(preprocessor.preprocess)
df["full_text_processed"] = df["full_text"].apply(preprocessor.preprocess)
df["word_count"] = df["content"].str.split().str.len()
df["char_count"] = df["content"].str.len()

print("Preprocessing complete.")
display(df[["article_id", "category", "title", "word_count", "content_processed"]].head())

## 4. Advanced Content Analysis Engine

This required final-project module improves the midterm classifier with confidence scores, probability distributions, cross-validation, and explanation-ready feature names.

In [ ]:
class AdvancedNewsClassifier:
    """Enhanced news classifier with confidence scoring and model comparison."""

    def __init__(self, max_features=8000, random_state=42):
        self.max_features = max_features
        self.random_state = random_state
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.85,
            sublinear_tf=True,
        )
        self.model = OneVsRestClassifier(
            LogisticRegression(
                max_iter=1000,
                solver="liblinear",
                C=4.0,
                class_weight="balanced",
                random_state=random_state,
            ),
            n_jobs=1,
        )
        self.feature_names_ = None
        self.results_ = None

    def _extra_features(self, texts):
        """Create non-negative sentiment and length features for classification."""
        sia = SentimentIntensityAnalyzer()
        rows = []
        for text in texts:
            text = "" if pd.isna(text) else str(text)
            scores = sia.polarity_scores(text[:5000])
            word_count = max(len(text.split()), 1)
            char_count = max(len(text), 1)
            rows.append([
                (scores["compound"] + 1) / 2,
                scores["pos"],
                scores["neu"],
                scores["neg"],
                min(word_count / 1000, 1),
                min(char_count / 7000, 1),
            ])
        return np.array(rows, dtype=float)

    def _build_matrix(self, texts, fit=False):
        """Combine TF-IDF features with sentiment and length features."""
        if fit:
            text_matrix = self.vectorizer.fit_transform(texts)
            self.feature_names_ = list(self.vectorizer.get_feature_names_out()) + [
                "compound_sentiment_scaled",
                "positive_score",
                "neutral_score",
                "negative_score",
                "word_count_scaled",
                "char_count_scaled",
            ]
        else:
            text_matrix = self.vectorizer.transform(texts)

        extra_matrix = csr_matrix(self._extra_features(texts))
        return hstack([text_matrix, extra_matrix]).tocsr().copy()

    def fit(self, texts, labels):
        """Train the enhanced classifier and compare multiple algorithms."""
        X = self._build_matrix(texts, fit=True)
        y = np.array(labels)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )

        classifiers = {
            "ComplementNB": ComplementNB(alpha=0.5),
            "LogisticRegression": self.model,
            "LinearSVC": LinearSVC(random_state=self.random_state, C=1.0, class_weight="balanced"),
        }

        results = []
        fitted = {}
        for name, clf in classifiers.items():
            print(f"Training {name}...")
            clf.fit(X_train, y_train)
            predictions = clf.predict(X_test)
            cv_scores = cross_val_score(clf, X_train, y_train, cv=2, scoring="accuracy")
            results.append({
                "Model": name,
                "Test Accuracy": accuracy_score(y_test, predictions),
                "CV Mean": cv_scores.mean(),
                "CV Std": cv_scores.std(),
            })
            fitted[name] = clf

        self.results_ = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)

        # Keep Logistic Regression as the production model because it provides probabilities and explanations.
        self.model = fitted["LogisticRegression"]
        self.X_test_ = X_test
        self.y_test_ = y_test
        self.test_predictions_ = self.model.predict(X_test)
        return self

    def predict_proba(self, texts):
        """Return class probabilities for one or more texts."""
        X = self._build_matrix(list(texts), fit=False)
        return self.model.predict_proba(X)

    def predict_article(self, title, content):
        """Predict category, confidence, and class probabilities for one article."""
        text = f"{title} {content}".strip()
        probabilities = self.predict_proba([text])[0]
        classes = self.model.classes_
        best_idx = int(np.argmax(probabilities))
        confidence = float(probabilities[best_idx])
        if confidence >= 0.70:
            confidence_label = "high"
        elif confidence >= 0.45:
            confidence_label = "medium"
        else:
            confidence_label = "low"
        return {
            "predicted_category": classes[best_idx],
            "confidence": confidence,
            "confidence_label": confidence_label,
            "category_probabilities": dict(zip(classes, probabilities.round(4))),
        }

    def explain_prediction(self, title, content, top_n=8):
        """Explain a Logistic Regression prediction using the strongest feature contributions."""
        text = f"{title} {content}".strip()
        X = self._build_matrix([text], fit=False)
        prediction = self.model.predict(X)[0]
        class_index = list(self.model.classes_).index(prediction)

        if hasattr(self.model, "coef_"):
            coefficients = self.model.coef_[class_index]
        elif hasattr(self.model, "estimators_"):
            coefficients = self.model.estimators_[class_index].coef_[0]
        else:
            return pd.DataFrame({"feature": ["Explanation unavailable"], "contribution": [0.0]})

        contributions = X.multiply(coefficients).toarray().ravel()
        top_indices = np.argsort(contributions)[-top_n:][::-1]
        return pd.DataFrame({
            "feature": [self.feature_names_[i] for i in top_indices if i < len(self.feature_names_)],
            "contribution": [contributions[i] for i in top_indices if i < len(self.feature_names_)],
        })


classifier = AdvancedNewsClassifier(max_features=config.tfidf_features, random_state=config.random_state)
classifier.fit(df["full_text_processed"], df["category"])

print("Classifier comparison:")
display(classifier.results_.round(4))

print("\nProduction model evaluation:")
print(classification_report(classifier.y_test_, classifier.test_predictions_))

cm = confusion_matrix(classifier.y_test_, classifier.test_predictions_, labels=classifier.model.classes_)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classifier.model.classes_, yticklabels=classifier.model.classes_)
plt.title("NewsBot 2.0 Classification Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 5. Topic Modeling and Content Discovery

The final project requires topic discovery. This section implements both LDA and NMF so the system can discover hidden themes beyond the original category labels.

In [ ]:
class TopicDiscoveryEngine:
    """Discover hidden topics using LDA and NMF."""

    def __init__(self, n_topics=8, max_features=5000, random_state=42):
        self.n_topics = n_topics
        self.random_state = random_state
        self.count_vectorizer = CountVectorizer(max_features=max_features, min_df=2, max_df=0.85, stop_words="english")
        self.tfidf_vectorizer = TfidfVectorizer(max_features=max_features, min_df=2, max_df=0.85, stop_words="english")
        self.lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=random_state, learning_method="batch")
        self.nmf_model = NMF(n_components=n_topics, random_state=random_state, init="nndsvda", max_iter=500)

    def fit(self, documents):
        """Train LDA and NMF topic models."""
        self.documents_ = list(documents)
        self.count_matrix_ = self.count_vectorizer.fit_transform(self.documents_)
        self.tfidf_matrix_ = self.tfidf_vectorizer.fit_transform(self.documents_)
        self.lda_topics_ = self.lda_model.fit_transform(self.count_matrix_)
        self.nmf_topics_ = self.nmf_model.fit_transform(self.tfidf_matrix_)
        return self

    def get_topic_words(self, method="nmf", n_words=10):
        """Return the top words for each topic."""
        if method == "lda":
            model = self.lda_model
            feature_names = self.count_vectorizer.get_feature_names_out()
        else:
            model = self.nmf_model
            feature_names = self.tfidf_vectorizer.get_feature_names_out()

        topic_rows = []
        for topic_id, weights in enumerate(model.components_):
            top_indices = weights.argsort()[-n_words:][::-1]
            words = [feature_names[i] for i in top_indices]
            topic_rows.append({"topic_id": topic_id, "top_words": ", ".join(words)})
        return pd.DataFrame(topic_rows)

    def transform_new_text(self, text):
        """Assign a dominant NMF topic to new text."""
        processed = preprocessor.preprocess(text)
        vector = self.tfidf_vectorizer.transform([processed])
        distribution = self.nmf_model.transform(vector)[0]
        topic_id = int(np.argmax(distribution))
        return {"dominant_topic": topic_id, "topic_strength": float(distribution[topic_id])}


topic_engine = TopicDiscoveryEngine(
    n_topics=config.topic_count,
    max_features=config.topic_features,
    random_state=config.random_state,
)
topic_engine.fit(df["content_processed"])

df["dominant_topic"] = np.argmax(topic_engine.nmf_topics_, axis=1)

print("NMF topic words:")
display(topic_engine.get_topic_words("nmf", n_words=10))

print("LDA topic words:")
display(topic_engine.get_topic_words("lda", n_words=10))

topic_category = pd.crosstab(df["dominant_topic"], df["category"])
plt.figure(figsize=(10, 6))
sns.heatmap(topic_category, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Discovered Topics by Original News Category")
plt.xlabel("Category")
plt.ylabel("Dominant Topic")
plt.tight_layout()
plt.show()

## 6. Advanced Sentiment Evolution Analysis

This section extends the midterm sentiment work by tracking sentiment across categories and generated analysis months. The BBC dataset does not provide verified publication dates, so `analysis_month` supports trend demonstration only.

In [ ]:
class SentimentEvolutionTracker:
    """Analyze article sentiment and track changes over categories and time periods."""

    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()

    def analyze_text(self, text):
        """Return VADER sentiment scores and a readable label."""
        scores = self.sia.polarity_scores("" if pd.isna(text) else str(text))
        if scores["compound"] >= 0.05:
            label = "positive"
        elif scores["compound"] <= -0.05:
            label = "negative"
        else:
            label = "neutral"
        return {**scores, "sentiment_label": label}

    def transform_dataset(self, frame, text_col="full_text"):
        """Add sentiment columns to the dataset."""
        rows = []
        for _, row in frame.iterrows():
            scores = self.analyze_text(row[text_col])
            rows.append({
                "article_id": row["article_id"],
                "category": row["category"],
                "analysis_month": row["analysis_month"],
                "compound": scores["compound"],
                "positive": scores["pos"],
                "neutral": scores["neu"],
                "negative": scores["neg"],
                "sentiment_label": scores["sentiment_label"],
            })
        self.sentiment_df_ = pd.DataFrame(rows)
        return self.sentiment_df_

    def category_summary(self):
        """Summarize sentiment by category."""
        return self.sentiment_df_.groupby("category").agg(
            avg_sentiment=("compound", "mean"),
            sentiment_std=("compound", "std"),
            positive_rate=("sentiment_label", lambda x: (x == "positive").mean()),
            negative_rate=("sentiment_label", lambda x: (x == "negative").mean()),
        ).round(4)


sentiment_tracker = SentimentEvolutionTracker()
sentiment_df = sentiment_tracker.transform_dataset(df)

print("Sentiment by category:")
display(sentiment_tracker.category_summary())

trend_df = sentiment_df.groupby(["analysis_month", "category"], as_index=False)["compound"].mean()
plt.figure(figsize=(12, 5))
sns.lineplot(data=trend_df, x="analysis_month", y="compound", hue="category", marker="o")
plt.title("Sentiment Evolution by Category (Generated Analysis Months)")
plt.xlabel("Analysis month")
plt.ylabel("Average VADER compound sentiment")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 7. Entity Relationship Mapping

This section expands the midterm NER work by mapping entity co-occurrences. This helps reveal which people, organizations, locations, and events appear together in news coverage.

In [ ]:
class EntityRelationshipMapper:
    """Extract named entities and build co-occurrence relationships."""

    def __init__(self, nlp_model, allowed_labels=None):
        self.nlp = nlp_model
        self.allowed_labels = allowed_labels or {"PERSON", "ORG", "GPE", "DATE", "MONEY", "PRODUCT", "EVENT"}

    def extract_from_doc(self, text):
        """Extract named entities from one text."""
        doc = self.nlp(str(text)[:5000])
        entities = []
        for ent in doc.ents:
            if ent.label_ in self.allowed_labels and len(ent.text.strip()) > 1:
                entities.append({
                    "text": ent.text.strip(),
                    "label": ent.label_,
                    "description": spacy.explain(ent.label_) or ent.label_,
                })
        return entities

    def fit(self, frame, text_col="full_text", max_articles=700):
        """Extract entities from a dataset sample and count entity relationships."""
        sample = frame.head(max_articles).copy()
        all_entities = []
        article_entities = []
        pair_counter = Counter()

        for _, row in sample.iterrows():
            entities = self.extract_from_doc(row[text_col])
            article_entities.append({
                "article_id": row["article_id"],
                "category": row["category"],
                "entities": entities,
            })

            for entity in entities:
                all_entities.append({
                    "article_id": row["article_id"],
                    "category": row["category"],
                    **entity,
                })

            relationship_entities = sorted(set(
                entity["text"] for entity in entities
                if entity["label"] in {"PERSON", "ORG", "GPE", "PRODUCT", "EVENT"}
            ))[:12]
            pair_counter.update(combinations(relationship_entities, 2))

        self.entities_df_ = pd.DataFrame(all_entities)
        self.article_entities_ = article_entities
        self.pair_counter_ = pair_counter
        return self

    def top_relationships(self, n=15):
        """Return the most common entity co-occurrences."""
        return pd.DataFrame([
            {"entity_1": pair[0], "entity_2": pair[1], "co_mentions": count}
            for pair, count in self.pair_counter_.most_common(n)
        ])

    def network_graph(self, n=20):
        """Create a NetworkX graph from top entity relationships."""
        graph = nx.Graph()
        for pair, count in self.pair_counter_.most_common(n):
            graph.add_edge(pair[0], pair[1], weight=count)
        return graph


entity_mapper = EntityRelationshipMapper(nlp)
entity_mapper.fit(df, max_articles=config.entity_article_limit)

print(f"Entities extracted: {len(entity_mapper.entities_df_)} from first {config.entity_article_limit} articles")
if not entity_mapper.entities_df_.empty:
    display(entity_mapper.entities_df_[["text", "label", "category"]].head(10))
    print("Top entity types:")
    display(entity_mapper.entities_df_["label"].value_counts().head(10).to_frame("mentions"))
    print("Top entity relationships:")
    display(entity_mapper.top_relationships(15))

    graph = entity_mapper.network_graph(n=20)
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(graph, seed=42, k=0.7)
    weights = [graph[u][v]["weight"] for u, v in graph.edges()]
    nx.draw_networkx_nodes(graph, pos, node_size=700, node_color="#9ecae1")
    nx.draw_networkx_edges(graph, pos, width=[max(w / 2, 1) for w in weights], alpha=0.5)
    nx.draw_networkx_labels(graph, pos, font_size=8)
    plt.title("Top Entity Co-occurrence Network")
    plt.axis("off")
    plt.show()

## 8. Language Understanding and Generation

This module adds extractive summarization, keyword extraction, semantic search, and automated insight generation. These features make NewsBot 2.0 more useful for a nontechnical user.

In [ ]:
class IntelligentSummarizer:
    """Create extractive summaries without paid language-model APIs."""

    def __init__(self, preprocessor, default_sentences=3):
        self.preprocessor = preprocessor
        self.default_sentences = default_sentences

    def split_sentences(self, text):
        """Split text into readable sentences."""
        sentences = re.split(r"(?<=[.!?])\s+", str(text).strip())
        return [sentence.strip() for sentence in sentences if len(sentence.strip().split()) >= 6]

    def summarize(self, text, n_sentences=None):
        """Score sentences by important word frequency and return top sentences in original order."""
        n_sentences = n_sentences or self.default_sentences
        sentences = self.split_sentences(text)
        if len(sentences) <= n_sentences:
            return " ".join(sentences)

        processed_tokens = self.preprocessor.preprocess(text).split()
        word_scores = Counter(processed_tokens)
        max_score = max(word_scores.values()) if word_scores else 1
        word_scores = {word: score / max_score for word, score in word_scores.items()}

        sentence_scores = []
        for index, sentence in enumerate(sentences):
            tokens = self.preprocessor.preprocess(sentence).split()
            if not tokens:
                continue
            score = sum(word_scores.get(token, 0) for token in tokens) / len(tokens)
            sentence_scores.append((index, score, sentence))

        top_sentences = sorted(sentence_scores, key=lambda x: x[1], reverse=True)[:n_sentences]
        top_sentences = sorted(top_sentences, key=lambda x: x[0])
        return " ".join(sentence for _, _, sentence in top_sentences)

    def keywords(self, text, n=8):
        """Return simple frequency-based keywords from processed text."""
        tokens = self.preprocessor.preprocess(text).split()
        return [word for word, _ in Counter(tokens).most_common(n)]


class SemanticSearchEngine:
    """Find similar articles and answer search-style queries with TF-IDF similarity."""

    def __init__(self, max_features=10000):
        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2, max_df=0.9)

    def fit(self, frame, text_col="full_text_processed"):
        """Fit the semantic search index."""
        self.frame_ = frame.reset_index(drop=True).copy()
        self.matrix_ = self.vectorizer.fit_transform(self.frame_[text_col])
        return self

    def search(self, query, top_n=5):
        """Return the most similar articles for a query."""
        processed_query = preprocessor.preprocess(query)
        query_vector = self.vectorizer.transform([processed_query])
        scores = cosine_similarity(query_vector, self.matrix_).ravel()
        top_indices = scores.argsort()[-top_n:][::-1]
        results = self.frame_.iloc[top_indices][["article_id", "title", "category"]].copy()
        results["similarity"] = scores[top_indices].round(4)
        return results

    def similar_to_article(self, article_index, top_n=5):
        """Find articles most similar to a selected article."""
        article_vector = self.matrix_[article_index]
        scores = cosine_similarity(article_vector, self.matrix_).ravel()
        scores[article_index] = -1
        top_indices = scores.argsort()[-top_n:][::-1]
        results = self.frame_.iloc[top_indices][["article_id", "title", "category"]].copy()
        results["similarity"] = scores[top_indices].round(4)
        return results


summarizer = IntelligentSummarizer(preprocessor, default_sentences=config.default_summary_sentences)
semantic_search = SemanticSearchEngine()
semantic_search.fit(df)

sample_article = df.iloc[0]
print("Sample summary:")
print(summarizer.summarize(sample_article["content"], n_sentences=3))
print("\nSample keywords:")
print(summarizer.keywords(sample_article["content"], n=10))
print("\nSemantic search demo: artificial intelligence and technology companies")
display(semantic_search.search("artificial intelligence and technology companies", top_n=5))

In [ ]:
class ContentEnhancer:
    """Generate business-friendly insights from multiple NewsBot components."""

    def __init__(self, classifier, summarizer, topic_engine, sentiment_tracker, entity_mapper):
        self.classifier = classifier
        self.summarizer = summarizer
        self.topic_engine = topic_engine
        self.sentiment_tracker = sentiment_tracker
        self.entity_mapper = entity_mapper

    def analyze_article(self, title, content):
        """Return an integrated analysis package for one article."""
        full_text = f"{title} {content}".strip()
        classification = self.classifier.predict_article(title, content)
        sentiment = self.sentiment_tracker.analyze_text(full_text)
        topic = self.topic_engine.transform_new_text(full_text)
        entities = self.entity_mapper.extract_from_doc(full_text)
        summary = self.summarizer.summarize(content, n_sentences=3)
        keywords = self.summarizer.keywords(full_text, n=8)

        insights = []
        insights.append(f"Predicted category is {classification['predicted_category']} with {classification['confidence_label']} confidence.")
        insights.append(f"Sentiment is {sentiment['sentiment_label']} with VADER score {sentiment['compound']:.3f}.")
        insights.append(f"Dominant discovered topic is Topic {topic['dominant_topic']}.")
        if entities:
            key_entities = ", ".join(sorted(set(entity["text"] for entity in entities[:6])))
            insights.append(f"Key entities include: {key_entities}.")
        insights.append(f"Useful keywords: {', '.join(keywords)}.")

        return {
            "title": title,
            "summary": summary,
            "classification": classification,
            "sentiment": sentiment,
            "topic": topic,
            "entities": entities,
            "keywords": keywords,
            "insights": insights,
        }


content_enhancer = ContentEnhancer(classifier, summarizer, topic_engine, sentiment_tracker, entity_mapper)

## 9. Multilingual Intelligence

This module demonstrates language detection, translation workflow design, and cross-language analysis. To keep the project free, the notebook uses `langdetect` for language identification and a transparent offline translation fallback for demonstration. If a real translation API becomes available later, this class can be extended without changing the rest of the system.

In [ ]:
class MultilingualProcessor:
    """Detect languages and support a transparent translation workflow."""

    def __init__(self):
        self.language_names = {
            "en": "English",
            "es": "Spanish",
            "fr": "French",
            "de": "German",
            "it": "Italian",
            "pt": "Portuguese",
        }
        self.keyword_translation = {
            # Spanish
            "tecnologia": "technology",
            "empresa": "company",
            "mercado": "market",
            "gobierno": "government",
            "deporte": "sport",
            "equipo": "team",
            "salud": "health",
            "economia": "business economy",
            "inteligencia artificial": "artificial intelligence",
            # French
            "technologie": "technology",
            "entreprise": "company",
            "marche": "market",
            "gouvernement": "government",
            "sport": "sport",
            "equipe": "team",
            "sante": "health",
            "economie": "business economy",
            "intelligence artificielle": "artificial intelligence",
        }

    def detect_language(self, text):
        """Detect the language of a text string."""
        try:
            code = detect(str(text)[:1000])
        except Exception:
            code = "unknown"
        return {"language_code": code, "language_name": self.language_names.get(code, code)}

    def translate_to_english(self, text, manual_translation=None):
        """Translate to English using a manual translation when provided, otherwise use keyword fallback."""
        if manual_translation:
            return manual_translation

        translated = str(text).lower()
        for source, target in self.keyword_translation.items():
            translated = translated.replace(source, target)
        return translated

    def process_multilingual_article(self, title, content, manual_translation=None):
        """Detect language, translate if needed, and prepare text for the English NLP pipeline."""
        full_text = f"{title} {content}".strip()
        language = self.detect_language(full_text)
        english_text = full_text
        translation_note = "No translation needed."

        if language["language_code"] != "en":
            english_text = self.translate_to_english(full_text, manual_translation=manual_translation)
            translation_note = "Translated with manual/offline fallback for project demonstration."

        return {
            "language": language,
            "english_text": english_text,
            "translation_note": translation_note,
        }

    def demo_cross_language_analysis(self, classifier, sentiment_tracker):
        """Run a small cross-language demonstration using manually translated sample articles."""
        examples = [
            {
                "title": "AI Company Announces New Cloud Tool",
                "content": "A technology company announced a new artificial intelligence product for business customers.",
                "manual_translation": None,
            },
            {
                "title": "Empresa de tecnologia anuncia nueva herramienta",
                "content": "La empresa dijo que la inteligencia artificial ayudara al mercado y a los clientes de negocios.",
                "manual_translation": "A technology company said artificial intelligence will help the market and business customers.",
            },
            {
                "title": "Entreprise de technologie annonce un outil",
                "content": "L'entreprise a dit que l'intelligence artificielle aidera le marche et les clients professionnels.",
                "manual_translation": "A technology company said artificial intelligence will help the market and business customers.",
            },
        ]

        rows = []
        for item in examples:
            processed = self.process_multilingual_article(item["title"], item["content"], item.get("manual_translation"))
            prediction = classifier.predict_article(item["title"], processed["english_text"])
            sentiment = sentiment_tracker.analyze_text(processed["english_text"])
            rows.append({
                "original_title": item["title"],
                "detected_language": processed["language"]["language_name"],
                "predicted_category": prediction["predicted_category"],
                "confidence": round(prediction["confidence"], 3),
                "sentiment": sentiment["sentiment_label"],
                "translation_note": processed["translation_note"],
            })
        return pd.DataFrame(rows)


multilingual_processor = MultilingualProcessor()
print("Cross-language demonstration:")
display(multilingual_processor.demo_cross_language_analysis(classifier, sentiment_tracker))

## 10. Conversational Interface

The final project requires natural language queries and interactive exploration. This section implements a lightweight conversational layer that can answer common NewsBot questions without paid chatbot APIs.

In [ ]:
class ConversationalInterface:
    """Rule-based conversational interface for querying NewsBot 2.0."""

    def __init__(self, frame, classifier, summarizer, semantic_search, topic_engine, sentiment_df, entity_mapper):
        self.frame = frame.reset_index(drop=True)
        self.classifier = classifier
        self.summarizer = summarizer
        self.semantic_search = semantic_search
        self.topic_engine = topic_engine
        self.sentiment_df = sentiment_df
        self.entity_mapper = entity_mapper
        self.history = []

    def classify_intent(self, query):
        """Classify user intent from keywords."""
        q = query.lower()
        if any(word in q for word in ["help", "what can you do", "commands"]):
            return "help"
        if "summarize" in q or "summary" in q:
            return "summarize"
        if "similar" in q or "search" in q or "find" in q:
            return "search"
        if "topic" in q or "theme" in q:
            return "topics"
        if "entity" in q or "entities" in q or "organization" in q:
            return "entities"
        if "positive" in q or "negative" in q or "sentiment" in q:
            return "sentiment"
        if "category" in q or "classify" in q:
            return "category"
        return "search"

    def _extract_article_number(self, query):
        """Extract an article number from a query, defaulting to the first article."""
        match = re.search(r"\b(\d+)\b", query)
        if match:
            return max(0, min(int(match.group(1)), len(self.frame) - 1))
        return 0

    def process_query(self, query):
        """Process a natural-language query and return a readable answer."""
        intent = self.classify_intent(query)
        self.history.append({"query": query, "intent": intent})

        if intent == "help":
            return (
                "Try: 'summarize article 3', 'search AI companies', "
                "'show topics', 'show negative sentiment by category', or 'show top entities'."
            )

        if intent == "summarize":
            article_index = self._extract_article_number(query)
            row = self.frame.iloc[article_index]
            return f"Summary for article {article_index} ({row['category']}): {self.summarizer.summarize(row['content'])}"

        if intent == "topics":
            topics = self.topic_engine.get_topic_words("nmf", n_words=8)
            return topics.to_string(index=False)

        if intent == "entities":
            if self.entity_mapper.entities_df_.empty:
                return "No entities were extracted."
            top_entities = self.entity_mapper.entities_df_["text"].value_counts().head(10)
            return "Top entities:\n" + top_entities.to_string()

        if intent == "sentiment":
            summary = self.sentiment_df.groupby("category")["compound"].mean().sort_values(ascending=False)
            return "Average sentiment by category:\n" + summary.round(3).to_string()

        if intent == "category":
            category_counts = self.frame["category"].value_counts()
            return "Dataset categories:\n" + category_counts.to_string()

        results = self.semantic_search.search(query, top_n=5)
        return "Search results:\n" + results.to_string(index=False)


conversation = ConversationalInterface(
    df, classifier, summarizer, semantic_search, topic_engine, sentiment_df, entity_mapper
)

for query in [
    "help",
    "summarize article 4",
    "search artificial intelligence companies",
    "show topics",
    "show sentiment by category",
    "show top entities",
]:
    print("\nUSER:", query)
    print("NEWSBOT:", conversation.process_query(query))

## 11. Complete NewsBot 2.0 Integrated System

This class brings the separate modules together into one reusable system. A user can paste an article and receive category prediction, confidence, topic, sentiment, summary, entities, multilingual handling, semantic matches, and insights.

In [ ]:
class NewsBot2IntegratedSystem:
    """Complete final-project NewsBot system."""

    def __init__(
        self,
        classifier,
        summarizer,
        topic_engine,
        sentiment_tracker,
        entity_mapper,
        semantic_search,
        multilingual_processor,
        content_enhancer,
    ):
        self.classifier = classifier
        self.summarizer = summarizer
        self.topic_engine = topic_engine
        self.sentiment_tracker = sentiment_tracker
        self.entity_mapper = entity_mapper
        self.semantic_search = semantic_search
        self.multilingual_processor = multilingual_processor
        self.content_enhancer = content_enhancer

    def process_article(self, title, content, manual_translation=None):
        """Process one article through the full NewsBot 2.0 pipeline."""
        multilingual = self.multilingual_processor.process_multilingual_article(
            title, content, manual_translation=manual_translation
        )
        analysis_text = multilingual["english_text"]
        enhanced = self.content_enhancer.analyze_article(title, analysis_text)
        similar_articles = self.semantic_search.search(analysis_text, top_n=3)
        explanation = self.classifier.explain_prediction(title, analysis_text, top_n=8)
        return {
            "multilingual": multilingual,
            "analysis": enhanced,
            "similar_articles": similar_articles,
            "classification_explanation": explanation,
        }


newsbot2 = NewsBot2IntegratedSystem(
    classifier=classifier,
    summarizer=summarizer,
    topic_engine=topic_engine,
    sentiment_tracker=sentiment_tracker,
    entity_mapper=entity_mapper,
    semantic_search=semantic_search,
    multilingual_processor=multilingual_processor,
    content_enhancer=content_enhancer,
)


test_articles = [
    {
        "title": "Microsoft Announces New AI Tools for Cloud Customers",
        "content": (
            "Microsoft announced a new artificial intelligence product for business customers. "
            "The company said the system will help organizations analyze data, summarize documents, "
            "and improve productivity across cloud services."
        ),
    },
    {
        "title": "Government Leaders Debate New Economic Policy",
        "content": (
            "Government leaders met today to debate a new economic policy. Officials said the plan "
            "could affect taxes, public spending, and business investment over the next year."
        ),
    },
    {
        "title": "Local Team Wins Championship After Dramatic Final",
        "content": (
            "The local team won the championship after a dramatic final match. Fans celebrated as the "
            "coach praised the players for their discipline, teamwork, and late-game scoring."
        ),
    },
]

for item in test_articles:
    result = newsbot2.process_article(item["title"], item["content"])
    print("\n" + "=" * 80)
    print("TITLE:", item["title"])
    print("CATEGORY:", result["analysis"]["classification"]["predicted_category"], 
          f"({result['analysis']['classification']['confidence']:.1%})")
    print("SENTIMENT:", result["analysis"]["sentiment"]["sentiment_label"],
          f"({result['analysis']['sentiment']['compound']:.3f})")
    print("SUMMARY:", result["analysis"]["summary"])
    print("INSIGHTS:")
    for insight in result["analysis"]["insights"]:
        print("-", insight)
    print("EXPLANATION:")
    display(result["classification_explanation"])

## 12. Optional Interactive Demo

This Gradio interface supports quick article testing in Colab. It does not require paid hosting. Run the cell and paste a new article to test the integrated system.

In [ ]:
try:
    import gradio as gr

    def gradio_analyze(title, content):
        """Return dashboard-friendly NewsBot 2.0 outputs."""
        if not content or len(content.strip()) < 20:
            return "Please paste a longer article.", "", "", ""
        result = newsbot2.process_article(title, content)
        analysis = result["analysis"]
        classification = analysis["classification"]
        category_text = (
            f"{classification['predicted_category']} "
            f"({classification['confidence']:.1%}, {classification['confidence_label']} confidence)"
        )
        sentiment_text = (
            f"{analysis['sentiment']['sentiment_label']} "
            f"(score {analysis['sentiment']['compound']:.3f})"
        )
        entities = "\n".join(
            f"- {entity['text']} ({entity['label']})" for entity in analysis["entities"][:10]
        ) or "No major entities found."
        insights = "\n".join(f"- {insight}" for insight in analysis["insights"])
        return category_text, sentiment_text, analysis["summary"], entities + "\n\n" + insights

    with gr.Blocks(title="NewsBot 2.0") as demo:
        gr.Markdown("# NewsBot 2.0 Final Project Demo")
        gr.Markdown("Paste a news article to analyze category, sentiment, summary, entities, and insights.")
        title_input = gr.Textbox(label="Headline")
        content_input = gr.Textbox(label="Article text", lines=10)
        analyze_button = gr.Button("Analyze")
        category_output = gr.Textbox(label="Predicted category")
        sentiment_output = gr.Textbox(label="Sentiment")
        summary_output = gr.Textbox(label="Summary", lines=4)
        insight_output = gr.Markdown(label="Entities and insights")
        analyze_button.click(
            gradio_analyze,
            inputs=[title_input, content_input],
            outputs=[category_output, sentiment_output, summary_output, insight_output],
        )

    print("Gradio demo is ready. Uncomment demo.launch(share=False) when you want to run it.")
    # demo.launch(share=False)
except Exception as error:
    print(f"Gradio setup skipped: {error}")

## Bonus Opportunity: Web Application Frontend (30 pts)

A free Flask web application frontend has been added in the `web_app/` folder. This bonus feature does not require paid APIs, API keys, or paid hosting. It lets a nontechnical user paste a news article into a browser and receive category prediction, confidence, sentiment, summary, entities, topic keywords, similar articles, and insights.

To run it locally from the project folder:

```bash
pip install -r requirements.txt
cd web_app
python app.py
```

Then open `http://127.0.0.1:5000` in a browser.

In [ ]:
from pathlib import Path

web_app_files = sorted(
    str(path.relative_to(Path.cwd()))
    for path in Path("web_app").rglob("*")
    if path.is_file() and "__pycache__" not in str(path)
)

print("Bonus Web Application Frontend files:")
for file in web_app_files:
    print("-", file)

## Bonus Web Application Frontend in Colab (Gradio)

The Flask app in `web_app/` is useful for a local portfolio-style web app, but Colab does not automatically include those project files. This Gradio interface runs directly inside Colab after the NewsBot 2.0 system has been trained in the previous cells.

No paid APIs, API keys, or ngrok setup are required. If you set `LAUNCH_COLAB_WEB_APP = True`, Gradio will create a temporary browser interface. If you use `share=True`, it also creates a temporary public link that works while the Colab runtime is active.

In [ ]:
try:
    import gradio as gr
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr


def colab_newsbot_analyze(title, content):
    """Analyze article text through the integrated NewsBot 2.0 system for Gradio."""
    if not content or len(content.strip()) < 20:
        empty_df = pd.DataFrame()
        return (
            "Please paste at least 20 characters of article text.",
            "",
            "",
            "",
            empty_df,
            empty_df,
            empty_df,
            empty_df,
        )

    result = newsbot2.process_article(title or "Untitled article", content)
    analysis = result["analysis"]
    classification = analysis["classification"]
    sentiment = analysis["sentiment"]

    category_text = (
        f"{classification['predicted_category']} "
        f"({classification['confidence']:.1%}, {classification['confidence_label']} confidence)"
    )
    sentiment_text = f"{sentiment['sentiment_label'].title()} (compound score {sentiment['compound']:.3f})"
    summary_text = analysis["summary"]
    insights_text = "\n".join(f"- {insight}" for insight in analysis["insights"])

    probabilities_df = pd.DataFrame([
        {"category": category, "probability": probability}
        for category, probability in classification["category_probabilities"].items()
    ]).sort_values("probability", ascending=False)

    entities_df = pd.DataFrame(analysis["entities"][:15])
    if entities_df.empty:
        entities_df = pd.DataFrame([{"text": "No major entities found", "label": "", "description": ""}])
    else:
        entities_df = entities_df[["text", "label", "description"]]

    similar_df = result["similar_articles"].copy()
    explanation_df = result["classification_explanation"].copy()

    topic_df = pd.DataFrame([{
        "dominant_topic": analysis["topic"]["dominant_topic"],
        "topic_strength": analysis["topic"]["topic_strength"],
        "keywords": ", ".join(analysis["keywords"]),
    }])

    return (
        category_text,
        sentiment_text,
        summary_text,
        insights_text,
        probabilities_df,
        entities_df,
        similar_df,
        explanation_df,
    )


with gr.Blocks(title="NewsBot 2.0 Colab Web App") as colab_demo:
    gr.Markdown("# NewsBot 2.0 Web Application Frontend")
    gr.Markdown(
        "Paste a news article to classify it, summarize it, analyze sentiment, "
        "extract entities, find similar articles, and generate insights."
    )
    title_input = gr.Textbox(label="Headline / Title", placeholder="Optional headline")
    content_input = gr.Textbox(label="Article Text", lines=12, placeholder="Paste a full news article here...")
    analyze_button = gr.Button("Analyze Article", variant="primary")

    category_output = gr.Textbox(label="Predicted Category")
    sentiment_output = gr.Textbox(label="Sentiment")
    summary_output = gr.Textbox(label="Summary", lines=5)
    insights_output = gr.Markdown(label="Insights")
    probabilities_output = gr.Dataframe(label="Category Probabilities")
    entities_output = gr.Dataframe(label="Named Entities")
    similar_output = gr.Dataframe(label="Similar Articles")
    explanation_output = gr.Dataframe(label="Classification Explanation")

    analyze_button.click(
        colab_newsbot_analyze,
        inputs=[title_input, content_input],
        outputs=[
            category_output,
            sentiment_output,
            summary_output,
            insights_output,
            probabilities_output,
            entities_output,
            similar_output,
            explanation_output,
        ],
    )


# Change this to True when you want to launch the web app in Colab.
# share=True creates a temporary public link while the runtime is active.
LAUNCH_COLAB_WEB_APP = False

if LAUNCH_COLAB_WEB_APP:
    colab_demo.launch(share=True, debug=False)
else:
    print("Colab web app is ready. Set LAUNCH_COLAB_WEB_APP = True and run this cell to launch it.")

## 13. Advanced Research Extension (Bonus Opportunity - 20 pts)

This bonus section adds a lightweight research-style extension: explainable prediction plus bias/framing analysis. The goal is not to claim perfect bias detection, but to show a transparent method for identifying emotionally loaded wording and explaining model decisions.

In [ ]:
class BiasAndFramingAnalyzer:
    """Transparent lexical bias/framing analyzer for news text."""

    def __init__(self):
        self.loaded_terms = {
            "shocking", "disaster", "outrage", "crisis", "scandal", "chaos", "furious",
            "radical", "extreme", "devastating", "alarming", "catastrophic", "controversial",
        }
        self.uncertainty_terms = {
            "alleged", "reportedly", "may", "might", "could", "claims", "suggests", "unclear",
        }
        self.authority_terms = {
            "expert", "official", "scientist", "analyst", "researcher", "minister", "agency",
        }

    def analyze(self, text):
        """Calculate a simple framing profile from transparent word lists."""
        tokens = re.findall(r"[a-z]+", str(text).lower())
        token_count = max(len(tokens), 1)
        counts = {
            "loaded_terms": sum(token in self.loaded_terms for token in tokens),
            "uncertainty_terms": sum(token in self.uncertainty_terms for token in tokens),
            "authority_terms": sum(token in self.authority_terms for token in tokens),
        }
        framing_score = (counts["loaded_terms"] + 0.5 * counts["uncertainty_terms"]) / token_count
        if framing_score >= 0.05:
            risk = "high"
        elif framing_score >= 0.02:
            risk = "medium"
        else:
            risk = "low"
        return {
            **counts,
            "token_count": token_count,
            "framing_score": round(framing_score, 4),
            "framing_risk": risk,
        }

    def evaluate_demo_cases(self):
        """Evaluate the analyzer on balanced and loaded sample wording."""
        examples = [
            {
                "case": "balanced",
                "text": "Officials announced a new policy after researchers reviewed the economic data.",
            },
            {
                "case": "loaded",
                "text": "Officials faced a shocking crisis after a controversial policy caused chaos and outrage.",
            },
            {
                "case": "uncertain",
                "text": "Analysts said the plan may affect jobs, but the long-term result remains unclear.",
            },
        ]
        rows = []
        for item in examples:
            rows.append({"case": item["case"], **self.analyze(item["text"])})
        return pd.DataFrame(rows)


bias_analyzer = BiasAndFramingAnalyzer()
demo_bias = bias_analyzer.evaluate_demo_cases()
display(demo_bias)

plt.figure(figsize=(8, 4))
sns.barplot(data=demo_bias, x="case", y="framing_score")
plt.title("Advanced Research Extension: Framing Score Demo")
plt.ylabel("Framing score")
plt.xlabel("Demo case")
plt.tight_layout()
plt.show()

research_text = (
    "The company announced a controversial restructuring plan. Analysts said the move could affect "
    "workers, but officials argued the decision may strengthen long-term business performance."
)

print("Bias/framing analysis:")
display(pd.DataFrame([bias_analyzer.analyze(research_text)]))

print("Explainable classification example:")
display(classifier.explain_prediction("Company Announces Restructuring Plan", research_text, top_n=10))

## 14. System Integration Testing and Validation Framework

The guidance notebook asks for comprehensive testing strategies, error handling, robustness checks, and performance awareness. This section adds an explicit test suite so the final project shows that the integrated system works as a complete pipeline, not only as separate components.

In [ ]:
class NewsBot2TestSuite:
    """Comprehensive testing framework for NewsBot 2.0."""

    def __init__(self, newsbot_system, frame):
        self.newsbot = newsbot_system
        self.frame = frame
        self.test_results = []

    def _record(self, test_name, passed, details):
        """Store one test result in a consistent format."""
        self.test_results.append({
            "test_name": test_name,
            "passed": bool(passed),
            "details": details,
        })

    def test_preprocessing(self):
        """Check that the preprocessing pipeline removes noise and keeps useful words."""
        sample = "Apple announced AI tools at https://example.com!!!"
        processed = preprocessor.preprocess(sample)
        passed = "http" not in processed and "apple" in processed and "tool" in processed
        self._record("preprocessing_noise_removal", passed, f"Processed text: {processed}")

    def test_classifier_prediction(self):
        """Check category prediction, probability output, and confidence bounds."""
        result = self.newsbot.classifier.predict_article(
            "Technology Company Launches AI Product",
            "A software company launched an artificial intelligence tool for cloud customers."
        )
        passed = (
            result["predicted_category"] in self.newsbot.classifier.model.classes_
            and 0 <= result["confidence"] <= 1
            and len(result["category_probabilities"]) == len(self.newsbot.classifier.model.classes_)
        )
        self._record("classifier_prediction_and_confidence", passed, str(result))

    def test_summarization(self):
        """Check that summarization returns a shorter readable summary."""
        text = self.frame.iloc[0]["content"]
        summary = self.newsbot.summarizer.summarize(text, n_sentences=2)
        passed = len(summary) > 0 and len(summary.split()) <= len(text.split())
        self._record("summarization_output", passed, summary[:250])

    def test_semantic_search(self):
        """Check that semantic search returns ranked article matches."""
        results = self.newsbot.semantic_search.search("technology companies and software", top_n=3)
        passed = len(results) == 3 and "similarity" in results.columns
        self._record("semantic_search_results", passed, results.to_string(index=False))

    def test_multilingual_workflow(self):
        """Check that language detection and translation workflow return usable English text."""
        result = self.newsbot.multilingual_processor.process_multilingual_article(
            "Empresa de tecnologia anuncia herramienta",
            "La empresa dijo que la inteligencia artificial ayudara al mercado.",
            manual_translation="A technology company said artificial intelligence will help the market."
        )
        passed = "language" in result and len(result["english_text"]) > 0
        self._record("multilingual_detection_translation", passed, str(result))

    def test_conversational_interface(self):
        """Check that the conversational layer gives a readable response."""
        response = conversation.process_query("show sentiment by category")
        passed = isinstance(response, str) and len(response) > 20 and "category" in response.lower()
        self._record("conversational_query_response", passed, response[:300])

    def test_integrated_pipeline(self):
        """Check that the final integrated system returns all major output groups."""
        result = self.newsbot.process_article(
            "Sports Team Wins Final Match",
            "The team won the final match after strong defense and late scoring by key players."
        )
        expected_keys = {"multilingual", "analysis", "similar_articles", "classification_explanation"}
        passed = expected_keys.issubset(result.keys()) and len(result["analysis"]["insights"]) >= 3
        self._record("integrated_pipeline_outputs", passed, str(list(result.keys())))

    def test_error_handling_small_input(self):
        """Check that the system handles very short input without crashing."""
        try:
            result = self.newsbot.process_article("Short update", "Markets moved today.")
            passed = "analysis" in result
            details = "Short input returned a valid analysis package."
        except Exception as error:
            passed = False
            details = f"Error: {error}"
        self._record("robustness_short_input", passed, details)

    def run_all_tests(self):
        """Run all system tests and return a summary DataFrame."""
        self.test_results = []
        self.test_preprocessing()
        self.test_classifier_prediction()
        self.test_summarization()
        self.test_semantic_search()
        self.test_multilingual_workflow()
        self.test_conversational_interface()
        self.test_integrated_pipeline()
        self.test_error_handling_small_input()
        return pd.DataFrame(self.test_results)


test_suite = NewsBot2TestSuite(newsbot2, df)
test_results_df = test_suite.run_all_tests()
print("NewsBot 2.0 integration test results:")
display(test_results_df)
print(f"Passed {test_results_df['passed'].sum()} out of {len(test_results_df)} tests.")

## 15. System Evaluation and Metrics

This section answers the final guidance question about how to evaluate the system's value. It combines model metrics, topic-model coverage, semantic-search behavior, integration-test results, and simple runtime monitoring.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import time


class NewsBot2Evaluator:
    """Comprehensive evaluation framework for NewsBot 2.0."""

    def __init__(self, newsbot_system, classifier, topic_engine, test_results):
        self.newsbot = newsbot_system
        self.classifier = classifier
        self.topic_engine = topic_engine
        self.test_results = test_results

    def evaluate_classification(self):
        """Return standard classification metrics for the production classifier."""
        y_true = self.classifier.y_test_
        y_pred = self.classifier.test_predictions_
        return {
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_precision": round(precision_score(y_true, y_pred, average="macro"), 4),
            "macro_recall": round(recall_score(y_true, y_pred, average="macro"), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
        }

    def evaluate_topic_modeling(self):
        """Return simple topic-model coverage metrics."""
        topic_counts = pd.Series(np.argmax(self.topic_engine.nmf_topics_, axis=1)).value_counts().sort_index()
        return {
            "number_of_topics": int(self.topic_engine.n_topics),
            "articles_with_topic_assignment": int(topic_counts.sum()),
            "smallest_topic_size": int(topic_counts.min()),
            "largest_topic_size": int(topic_counts.max()),
        }

    def evaluate_testing_quality(self):
        """Summarize the integration test suite."""
        return {
            "tests_run": int(len(self.test_results)),
            "tests_passed": int(self.test_results["passed"].sum()),
            "test_pass_rate": round(float(self.test_results["passed"].mean()), 4),
        }

    def evaluate_runtime(self, runs=3):
        """Estimate average processing time for the integrated article pipeline."""
        sample_title = "Technology Company Announces New AI Service"
        sample_content = (
            "A technology company announced a new artificial intelligence service for business customers. "
            "The tool summarizes documents, identifies important entities, and helps teams search information faster."
        )
        durations = []
        for _ in range(runs):
            start = time.perf_counter()
            self.newsbot.process_article(sample_title, sample_content)
            durations.append(time.perf_counter() - start)
        return {
            "runtime_tests": runs,
            "avg_seconds_per_article": round(float(np.mean(durations)), 4),
            "max_seconds_per_article": round(float(np.max(durations)), 4),
        }

    def build_evaluation_report(self):
        """Combine all evaluation results into one table."""
        rows = []
        for group_name, metrics in [
            ("classification", self.evaluate_classification()),
            ("topic_modeling", self.evaluate_topic_modeling()),
            ("integration_tests", self.evaluate_testing_quality()),
            ("runtime", self.evaluate_runtime()),
        ]:
            for metric, value in metrics.items():
                rows.append({"evaluation_area": group_name, "metric": metric, "value": value})
        return pd.DataFrame(rows)


evaluator = NewsBot2Evaluator(newsbot2, classifier, topic_engine, test_results_df)
evaluation_report_df = evaluator.build_evaluation_report()
print("NewsBot 2.0 evaluation report:")
display(evaluation_report_df)

## 16. Final System Evaluation Checklist

This section summarizes how the notebook addresses the required final-project modules.

In [ ]:
final_checklist = pd.DataFrame({
    "Requirement": [
        "Advanced classification with confidence scoring",
        "Topic modeling with LDA and NMF",
        "Sentiment analysis and trend tracking",
        "Entity relationship mapping",
        "Text summarization",
        "Semantic search",
        "Multilingual language detection and translation workflow",
        "Conversational natural-language query interface",
        "Integrated NewsBot 2.0 system class",
        "System integration testing and robustness checks",
        "Evaluation metrics and runtime monitoring",
        "Advanced research extension for explainability and framing analysis",
        "Web application frontend bonus",
    ],
    "Implemented": [True] * 13,
    "Notebook Section": [
        "4",
        "5",
        "6",
        "7",
        "8",
        "8",
        "9",
        "10",
        "11",
        "14",
        "15",
        "13",
        "Bonus Web App",
    ],
})

display(final_checklist)

print("NewsBot 2.0 coding foundation is ready for final documentation and presentation materials.")
